In [1]:
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters langchain-openai faiss-cpu pypdf

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_classic.chains import RetrievalQA
from langchain_community.document_loaders import PyPDFLoader

In [3]:
loader = PyPDFLoader("https://cs229.stanford.edu/main_notes.pdf")
docs = loader.load()

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
documents= text_splitter.split_documents(docs)

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
vectorstore = FAISS.from_documents(documents, embeddings)

In [7]:
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

In [ ]:
llm = ChatOpenAI()

In [9]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
     return_source_documents=True
)

In [10]:
query = "What is supervised learning?"
result = qa_chain({"query": query})
print(result["result"])

/tmp/ipykernel_2920/389418458.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  result = qa_chain({"query": query})


Supervised learning is a type of machine learning where an algorithm learns a function from labeled training data. This training data consists of input variables (x) and corresponding output variables (y), also known as the target variable. The learning algorithm uses this data to produce a hypothesis (h) that can predict the output (y) for new inputs (x).

Supervised learning problems are typically categorized into two types:
*   **Regression problems:** Occur when the target variable (y) is continuous (e.g., predicting the price of a house based on its living area).
*   **Classification problems:** Occur when the target variable (y) can take on only a small number of discrete values (e.g., predicting whether a dwelling is a house or an apartment).


In [11]:
for doc in result["source_documents"]:
    print(f"- {doc.page_content[:150]}...")

- Part I
Supervised learning
5...
- decision boundary it falls, and makes its prediction accordingly.
Here’s a diﬀerent approach. First, looking at elephants, we can build a
model of wha...
- 7
function h is called a hypothesis. Seen pictorially, the process is therefore
like this:
Training 
    set
 house.)
(living area of
Learning 
algori...


#### Advanced RAG with Langchain

In [12]:
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

In [19]:
memory = ConversationBufferMemory(
    memory_key='chat_history',
    return_messages=True,
    output_key="answer"
)

In [20]:
conversational_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory
)

In [21]:
response1 = conversational_chain.invoke({"question": "Hello, My Name is Mohsin"})

print(response1["answer"])

Hello Mohsin! It's nice to meet you. How can I help you today?


In [22]:
response2 = conversational_chain.invoke({"question": "Tell me what is My name?"})

print(response2["answer"])

I don't know, the provided context does not contain any information about your name.
